# Step 2 — Feature Extraction

This notebook extracts two feature blocks per sequence:

1. **PSCV (Position-Specific Composition-Variant)** — 153-d hand-crafted descriptor
2. **PLM (Protein Language Model) embeddings** — ESM-2 `t33_650M` mean-pooled (1280-d)

Both blocks are computed for the **training set** (70%) and the **independent test set** (30%) created in Step 1, then concatenated and saved to `features.npz`.

**This is the slow step.** PSCV takes minutes on CPU; ESM-2 needs a GPU runtime. Output is cached, so re-running is fast if `features.npz` exists.

## Cell 1 — Configuration

In [ ]:
WORK_DIR = '/content/drive/MyDrive/pscv_work'

PLM_MODEL = 'facebook/esm2_t33_650M_UR50D'
PLM_DIM   = 1280
MAX_LEN   = 1022
BATCH_SIZE = 4

import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('Work directory:', WORK_DIR)
print('PLM model     :', PLM_MODEL, f'({PLM_DIM}-d)')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device        :', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: PLM embedding on CPU will be very slow. Switch to a GPU runtime.')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q biopython transformers accelerate sentencepiece

## Cell 3 — Load partitioned data from Step 1

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv(os.path.join(WORK_DIR, 'train.csv'))
indep_test_df = pd.read_csv(os.path.join(WORK_DIR, 'independent_test.csv'))

print(f'Training set     : {len(train_df)} sequences')
print(f'Independent test : {len(indep_test_df)} sequences')

## Cell 4 — PSCV feature extractor (153-d)

In [ ]:
import math
import numpy as np

AA_ORDER = ['A','C','D','E','F','G','H','I','K','L',
            'M','N','P','Q','R','S','T','V','W','Y','X']
AA_INDEX = {aa: i for i, aa in enumerate(AA_ORDER)}
N_AA = len(AA_ORDER)
ALLOWED = set(AA_ORDER)


def seq_to_mat(seq):
    mat = np.zeros((N_AA, N_AA), dtype=np.float64)
    for k in range(len(seq) - 1):
        a, b = seq[k], seq[k+1]
        if a in AA_INDEX and b in AA_INDEX:
            mat[AA_INDEX[a], AA_INDEX[b]] += 1
    return mat


def prim(seq):
    mat = np.zeros((N_AA, N_AA), dtype=np.float64)
    for i, aa1 in enumerate(AA_ORDER):
        first = -1
        for x, ch in enumerate(seq):
            if ch == aa1:
                first = x + 1
                break
        if first == -1:
            continue
        for j, aa2 in enumerate(AA_ORDER):
            if i == j: continue
            s = 0
            for y, ch in enumerate(seq):
                if ch == aa2:
                    s += (y + 1) - first
            mat[i, j] = s
    return mat


def frequency_vec(seq):
    fv = [0] * N_AA
    for ch in seq:
        if ch in AA_INDEX:
            fv[AA_INDEX[ch]] += 1
    return fv


def aapiv(seq):
    encoder = ['X'] + AA_ORDER[:-1]
    apv = [0] * N_AA
    for i in range(N_AA):
        s = 0
        for j, ch in enumerate(seq):
            if ch == encoder[i]:
                s += j + 1
        apv[i] = s
    return apv[1:] + apv[:1]


def raw_moments(mat, order=3):
    n = mat.shape[0]
    out = []
    idx = np.arange(1, n+1, dtype=np.float64)
    for i in range(order+1):
        for j in range(order+1):
            if i + j <= order:
                out.append(float((idx**i) @ mat @ (idx**j)))
    return out


def central_moments(mat, order, xbar, ybar):
    n = mat.shape[0]
    out = []
    idx = np.arange(1, n+1, dtype=np.float64)
    for i in range(order+1):
        for j in range(order+1):
            if i + j <= order:
                out.append(float(((idx - xbar)**i) @ mat @ ((idx - ybar)**j)))
    return out


def _poch(a, k):
    ans = 1.0
    for i in range(k):
        ans *= (a + i)
    return ans


def _gamma(x):
    return math.exp(math.lgamma(x))


def _hahn_processor(x, n, N):
    rho = _gamma(n+1.0) * _gamma(n+1.0) * _poch(n+1.0, N)
    if rho < 0: rho = 0.0
    ans1 = _poch(N-1.0, n) * _poch(N-1.0, n)
    ans2 = 0.0
    for k in range(n+1):
        ans2 += ((-1.0)**k) * (_poch(-n, k) * _poch(-x, k) * _poch(2*N - n - 1.0, k))
    return (ans1 + ans2) * math.sqrt(rho)


def _hahn_moment(m, n, N, mat):
    val = 0.0
    for x in range(N):
        for y in range(N):
            val += mat[x, y] * _hahn_processor(x, m, N) * _hahn_processor(x, n, N)
    return val


def hahn_moments(mat, order=3):
    N = mat.shape[0]
    out = []
    for i in range(order+1):
        for j in range(order+1):
            if i + j <= order:
                out.append(_hahn_moment(i, j, N, mat))
    return out


def pscv_features(seq, order=3):
    seq = seq.upper()
    seq = ''.join(ch if ch in ALLOWED else 'X' for ch in seq)
    fv = []
    M = seq_to_mat(seq)
    rm = raw_moments(M, order); fv.extend(rm)
    fv.extend(central_moments(M, order, rm[4], rm[1]))
    fv.extend(hahn_moments(M, order))
    fv.extend(frequency_vec(seq))
    P = prim(seq)
    rmP = raw_moments(P, order); fv.extend(rmP)
    fv.extend(central_moments(P, order, rmP[4], rmP[1]))
    fv.extend(hahn_moments(P, order))
    fv.extend(aapiv(seq))
    R = prim(seq[::-1])
    rmR = raw_moments(R, order); fv.extend(rmR)
    fv.extend(central_moments(R, order, rmR[4], rmR[1]))
    fv.extend(hahn_moments(R, order))
    fv.extend(aapiv(seq[::-1]))
    arr = np.asarray(fv, dtype=np.float64)
    assert arr.shape[0] == 153, f'expected 153, got {arr.shape[0]}'
    return arr


def pscv_group_indices():
    return {
        'seqMat_raw':     slice(0,   10),
        'seqMat_central': slice(10,  20),
        'seqMat_hahn':    slice(20,  30),
        'freq_vec':       slice(30,  51),
        'PRIM_raw':       slice(51,  61),
        'PRIM_central':   slice(61,  71),
        'PRIM_hahn':      slice(71,  81),
        'AAPIV':          slice(81,  102),
        'RPRIM_raw':      slice(102, 112),
        'RPRIM_central':  slice(112, 122),
        'RPRIM_hahn':     slice(122, 132),
        'RAAPIV':         slice(132, 153),
    }


_test = pscv_features('MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ')
print('PSCV smoke test OK, feature dim =', _test.shape[0])

## Cell 5 — Extract PSCV features

In [ ]:
import time

def extract_pscv_batch(df, name):
    feats = np.zeros((len(df), 153), dtype=np.float32)
    t0 = time.time()
    for i, seq in enumerate(df['sequence'].values):
        if i % 50 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-6)
            eta = (len(df) - i) / max(rate, 1e-6)
            print(f'  PSCV [{name}] {i:5d}/{len(df)}  ({rate:5.1f} seq/s, ETA {eta:5.0f}s)')
        feats[i] = pscv_features(seq).astype(np.float32)
    print(f'  PSCV [{name}] done. {len(df)} sequences in {time.time()-t0:.1f}s')
    return feats


print('Extracting PSCV for training set...')
X_pscv_train = extract_pscv_batch(train_df, 'train')

print('\nExtracting PSCV for independent test set...')
X_pscv_indep = extract_pscv_batch(indep_test_df, 'indep_test')

print('\nShapes:')
print('  X_pscv_train:', X_pscv_train.shape)
print('  X_pscv_indep:', X_pscv_indep.shape)

## Cell 6 — Load ESM-2 model

In [ ]:
from transformers import AutoTokenizer, AutoModel

print(f'Loading {PLM_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(PLM_MODEL, do_lower_case=False)
plm_model = AutoModel.from_pretrained(PLM_MODEL).to(DEVICE).eval()
print(f'Loaded. Parameters: {sum(p.numel() for p in plm_model.parameters())/1e6:.1f}M')

## Cell 7 — Mean-pooled ESM-2 embedding extraction

In [ ]:
def _clean_for_plm(seq):
    seq = seq.upper()
    return ''.join(('X' if c in 'UZOB' else c) for c in seq)


def _windows(seq, max_len):
    if len(seq) <= max_len:
        return [seq]
    return [seq[i:i+max_len] for i in range(0, len(seq), max_len)]


@torch.no_grad()
def embed_sequences(sequences, name=''):
    out = np.zeros((len(sequences), PLM_DIM), dtype=np.float32)
    t0 = time.time()
    for n, raw_seq in enumerate(sequences):
        if n % 25 == 0 and n > 0:
            elapsed = time.time() - t0
            rate = n / elapsed
            eta = (len(sequences) - n) / max(rate, 1e-6)
            print(f'  PLM [{name}] {n:5d}/{len(sequences)}  ({rate:5.2f} seq/s, ETA {eta:5.0f}s)')
        seq = _clean_for_plm(raw_seq)
        wins = _windows(seq, MAX_LEN)
        win_means = []
        for i in range(0, len(wins), BATCH_SIZE):
            batch = wins[i:i+BATCH_SIZE]
            enc = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=MAX_LEN + 2)
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            hs = plm_model(**enc).last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1).float()
            pooled = ((hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)).cpu().numpy()
            win_means.append(pooled)
        out[n] = np.concatenate(win_means, axis=0).mean(axis=0)
    print(f'  PLM [{name}] done. {len(sequences)} sequences in {time.time()-t0:.1f}s')
    return out


print('Extracting ESM-2 embeddings for training set...')
X_plm_train = embed_sequences(train_df['sequence'].tolist(), 'train')

print('\nExtracting ESM-2 embeddings for independent test set...')
X_plm_indep = embed_sequences(indep_test_df['sequence'].tolist(), 'indep_test')

print('\nShapes:')
print('  X_plm_train:', X_plm_train.shape)
print('  X_plm_indep:', X_plm_indep.shape)

del plm_model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Cell 8 — Concatenate blocks and save

In [ ]:
X_train = np.concatenate([X_pscv_train, X_plm_train], axis=1).astype(np.float32)
X_indep = np.concatenate([X_pscv_indep, X_plm_indep], axis=1).astype(np.float32)
y_train = train_df['label'].values.astype(np.int64)
y_indep = indep_test_df['label'].values.astype(np.int64)

block_sizes = np.array([X_pscv_train.shape[1], X_plm_train.shape[1]])
block_names = np.array(['PSCV', PLM_MODEL.split('/')[-1]])

print('Final feature matrices:')
print('  X_train:', X_train.shape, '  y_train:', y_train.shape)
print('  X_indep:', X_indep.shape, '  y_indep:', y_indep.shape)
print('  block_sizes:', block_sizes.tolist())
print('  block_names:', block_names.tolist())

out_path = os.path.join(WORK_DIR, 'features.npz')
np.savez_compressed(
    out_path,
    X_train=X_train,
    y_train=y_train,
    X_indep=X_indep,
    y_indep=y_indep,
    block_sizes=block_sizes,
    block_names=block_names,
)
print(f'\nSaved -> {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')
print('\nStep 2 complete. Open Step 3 (classification) next.')